# Lesson 27b: ANOVA in Multiple Linear Regression

ANOVA (analysis of variance) splits the variation in y into the part a model explains and the part it leaves over. We then use the **bootstrap** from Lesson 25 to ask whether a model's improvement is real or could be due to chance:

1. **Recap: four models.** Refit the hybrid-car models from Lessons 26–27.
2. **Sums of squares.** Split the variation in MSRP into SST = SSR + SSE.
3. **Bootstrap R².** How much would R² change with a different sample? Does the model beat the mean?
4. **Class means.** Compare mean MSRP across car classes (one-way ANOVA).
5. **Bootstrap the improvement.** Does car class help beyond acceleration and mpg?
6. **Class coefficients.** Which classes differ, holding acceleration and mpg fixed?
7. **Noise feature.** Watch the bootstrap say "no", and see why we don't bootstrap the R² gain.

Section names match the **Jupyter demo** tags on the slides.

[Acknowledgments Page](https://ds100.org/fa23/acks/)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

rng = np.random.default_rng(3022)   # fixed seed so results match the slides

## 1. Recap: four models

Same data and features as Lesson 27: acceleration, mpg, their transformations, and one-hot encoded car class (drop-first, so Compact is the baseline).

In [ ]:
hybrid = pd.read_csv('data/hybrid.csv')
n = len(hybrid)

hybrid['acc_sq'] = hybrid['acceleration'] ** 2
hybrid['mpg_sqrt'] = np.sqrt(hybrid['mpg'])
hybrid['noise'] = rng.normal(size=n)          # a useless feature for Section 7

ohe = OneHotEncoder(drop='first')
ohe.fit(hybrid[['class']])
class_names = list(ohe.get_feature_names_out())
cars = hybrid.join(pd.DataFrame(ohe.transform(hybrid[['class']]).toarray(),
                                columns=class_names, index=hybrid.index))

quant = ['acceleration', 'mpg', 'acc_sq', 'mpg_sqrt']
models = {
    'SLR':    ['acceleration'],
    'MLR V1': ['acceleration', 'mpg'],
    'MLR V2': quant,
    'MLR V3': quant + class_names,
}
n

A helper that fits a model and returns its sums of squares, R² and adjusted R². We'll use it throughout.

In [ ]:
def fit_stats(data, features):
    """Fit least squares (with intercept) on data; return SST, SSR, SSE, R^2 and adjusted R^2."""
    X, y = data[features], data['msrp']
    y_hat = LinearRegression().fit(X, y).predict(X)
    sst = np.sum((y - y.mean()) ** 2)
    ssr = np.sum((y_hat - y.mean()) ** 2)
    sse = np.sum((y - y_hat) ** 2)
    r2 = ssr / sst
    p, n_rows = len(features), len(y)
    adj_r2 = 1 - (1 - r2) * (n_rows - 1) / (n_rows - p - 1)
    return {'SST': sst, 'SSR': ssr, 'SSE': sse, 'R2': r2, 'adj_R2': adj_r2}

summary = pd.DataFrame({name: fit_stats(cars, f) for name, f in models.items()}).T
summary['RMSE'] = np.sqrt(summary['SSE'] / n)
summary[['RMSE', 'R2', 'adj_R2']].round(3)

Every added feature raises R². Is each improvement more than chance?

## 2. Sums of squares

For least squares with an intercept, **SST = SSR + SSE**, and **R² = SSR / SST**.

In [ ]:
ss = summary[['SST', 'SSR', 'SSE']] / 1e6          # millions of $^2
ss['SSR + SSE'] = ss['SSR'] + ss['SSE']
ss['R2'] = summary['R2']
ss.round(3)

SST is the same for every model (it doesn't depend on the features), and SSR + SSE matches it exactly. Adding features moves variation from SSE into SSR.

**Check:** R² is also Var(ŷ) / Var(y), the coefficient of determination from Lesson 24.

In [ ]:
y_hat_v2 = LinearRegression().fit(cars[quant], cars['msrp']).predict(cars[quant])
np.var(y_hat_v2) / np.var(cars['msrp']), summary.loc['MLR V2', 'R2']

## 3. Bootstrap R²

Our 153 cars are one sample. A different sample would give a different R². To see how much it could vary, bootstrap the rows as in Lesson 25:

1. Resample the cars (rows) with replacement.
2. Refit the model to the resample and compute the statistic.
3. Repeat many times; the middle 95% is an approximate 95% confidence interval.

With only 4 minivans, an occasional resample has no car from some class. We skip those resamples, since the class models can't be fit on them.

In [ ]:
def bootstrap(statistic, data, B=2000):
    """Resample the rows of data B times; return statistic(resample) for each one."""
    results = []
    while len(results) < B:
        resample = data.sample(len(data), replace=True, random_state=rng)
        if resample['class'].nunique() < data['class'].nunique():
            continue                                   # a class is missing: skip this resample
        results.append(statistic(resample))
    return np.array(results)

def ci95(values):
    return np.percentile(values, [2.5, 97.5], axis=0)

In [ ]:
def r2_and_adj(data):
    s = fit_stats(data, quant)
    return [s['R2'], s['adj_R2']]

boot_v2 = bootstrap(r2_and_adj, cars)
print('R^2 (MLR V2):         ', summary.loc['MLR V2', 'R2'].round(3), '  95% CI:', ci95(boot_v2[:, 0]).round(3))
print('adjusted R^2 (MLR V2):', summary.loc['MLR V2', 'adj_R2'].round(3), '  95% CI:', ci95(boot_v2[:, 1]).round(3))

In [ ]:
plt.hist(boot_v2[:, 1], bins=40, edgecolor='white')
lo, hi = ci95(boot_v2[:, 1])
plt.axvline(lo, color='goldenrod'); plt.axvline(hi, color='goldenrod')
plt.axvline(0, color='k', ls='--', label='0: no better than the mean')
plt.xlim(-0.05, 1); plt.xlabel('bootstrapped adjusted R²'); plt.legend();

**Does the model beat the mean?** Use **adjusted R²**, not R². Even useless features give R² > 0 in a sample, but adjusted R² is about 0 on average when the features don't help. The 95% CI for adjusted R² is far above 0, so we have strong evidence that MLR V2 beats the constant model.

## 4. Class means

Does mean MSRP differ across car classes? This is the question **one-way ANOVA** answers.

In [ ]:
order = list(cars.groupby('class')['msrp'].mean().sort_values().index)
plt.figure(figsize=(9, 4))
sns.boxplot(data=cars, x='class', y='msrp', order=order, color='lightsteelblue')
sns.stripplot(data=cars, x='class', y='msrp', order=order, size=3, color='k', alpha=0.5)
plt.axhline(cars['msrp'].mean(), ls='--', color='goldenrod', label='overall mean')
plt.legend();

One-way ANOVA is regression on the one-hot class columns alone: each prediction is that class's mean, so SSR = between-class variation and SSE = within-class variation.

In [ ]:
fit_stats(cars, class_names)

Bootstrap each class's mean MSRP:

In [ ]:
boot_means = bootstrap(lambda d: d.groupby('class')['msrp'].mean()[order].to_numpy(), cars)
lo, hi = ci95(boot_means)
class_ci = pd.DataFrame({'n': cars['class'].value_counts()[order],
                         'mean': cars.groupby('class')['msrp'].mean()[order],
                         'CI low': lo, 'CI high': hi}).round(0)
class_ci

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.errorbar(class_ci['mean'], range(len(order)),
             xerr=[class_ci['mean'] - class_ci['CI low'], class_ci['CI high'] - class_ci['mean']],
             fmt='o', capsize=4)
plt.yticks(range(len(order)), order)
plt.axvline(cars['msrp'].mean(), ls='--', color='goldenrod', label='overall mean')
plt.xlabel('mean MSRP with bootstrap 95% CI'); plt.legend();

The Two Seater and Large intervals are far apart, so at least two class means differ. One number for the overall question: bootstrap the class-only model's adjusted R².

In [ ]:
boot_class = bootstrap(lambda d: fit_stats(d, class_names)['adj_R2'], cars)
print('adjusted R^2 (class only):', fit_stats(cars, class_names)['adj_R2'].round(3), '  95% CI:', ci95(boot_class).round(3))

## 5. Bootstrap the improvement

Reduced model: MLR V2 (4 quantitative features). Full model: MLR V3 (the same 4 + the 6 class columns). They're **nested**.

In each resample, fit both models and record how much adjusted R² goes up.

In [ ]:
def adj_gain(data, reduced, full):
    return fit_stats(data, full)['adj_R2'] - fit_stats(data, reduced)['adj_R2']

gain_obs = adj_gain(cars, quant, quant + class_names)
boot_gain = bootstrap(lambda d: adj_gain(d, quant, quant + class_names), cars)
print('gain in adjusted R^2:', gain_obs.round(3), '  95% CI:', ci95(boot_gain).round(3))

In [ ]:
plt.hist(boot_gain, bins=40, edgecolor='white')
lo, hi = ci95(boot_gain)
plt.axvline(lo, color='goldenrod'); plt.axvline(hi, color='goldenrod')
plt.axvline(0, color='k', ls='--', label='0: no improvement')
plt.xlabel('bootstrapped gain in adjusted R² from adding class'); plt.legend();

The whole interval is above 0: strong evidence that car class explains MSRP beyond acceleration and mpg.

## 6. Class coefficients

Which classes differ? Bootstrap the 6 class coefficients in MLR V3, exactly as Lesson 25 bootstrapped the slope. Each coefficient compares a class to Compact cars **with the same acceleration and mpg**.

In [ ]:
full_features = quant + class_names
coef_obs = LinearRegression().fit(cars[full_features], cars['msrp']).coef_[len(quant):]
boot_coefs = bootstrap(lambda d: LinearRegression().fit(d[full_features], d['msrp']).coef_[len(quant):], cars)
lo, hi = ci95(boot_coefs)
coef_ci = pd.DataFrame({'coefficient': coef_obs, 'CI low': lo, 'CI high': hi}, index=class_names).round(0)
coef_ci['CI excludes 0'] = (coef_ci['CI low'] > 0) | (coef_ci['CI high'] < 0)
coef_ci

Three classes (Large, Pickup Truck, Two Seater) have intervals that exclude 0. With 6 intervals, keep in mind that some could exclude 0 by chance.

## 7. Noise feature

Add a column of random noise to MLR V2. It can't really help, so a good method should say "no".

In [ ]:
def gains(data):
    red, full = fit_stats(data, quant), fit_stats(data, quant + ['noise'])
    noise_coef = LinearRegression().fit(data[quant + ['noise']], data['msrp']).coef_[-1]
    return [full['R2'] - red['R2'], full['adj_R2'] - red['adj_R2'], noise_coef]

obs = gains(cars)
boot_noise = bootstrap(gains, cars)
for k, name in enumerate(['gain in R^2', 'gain in adjusted R^2', 'noise coefficient']):
    print(f'{name:22s} observed: {obs[k]:10.4f}   95% CI: {ci95(boot_noise[:, k]).round(4)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, k, label in zip(axes, [0, 1], ['gain in R²', 'gain in adjusted R²']):
    ax.hist(boot_noise[:, k], bins=40, edgecolor='white')
    ax.axvline(0, color='k', ls='--')
    ax.set_xlabel(f'bootstrapped {label} from adding noise')

- The **gain in R²** can never be negative (R² never goes down when a feature is added), so its CI can never include values below 0. It can't tell us "no".
- The **gain in adjusted R²** and the **noise coefficient** both have CIs that contain 0: no evidence the noise helps.

**Takeaway:** to decide whether features help, bootstrap the gain in adjusted R² (for a group of features) or a coefficient (for one feature), never the gain in R².